In [ ]:
import pandas as pd
import numpy as np
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import os
import json
import pickle
import re

In [ ]:
data_dir = os.path.join("..", "data")
preprocess_data_dir = os.path.join(data_dir, "preprocess")
cluster_data_dir = os.path.join(data_dir, "cluster")
if not os.path.exists(cluster_data_dir):
    os.makedirs(cluster_data_dir)

In [ ]:
Seqs:pd.DataFrame = pd.read_pickle(os.path.join(preprocess_data_dir, "all_protein_df.pkl"))
all_fasta = os.path.join(cluster_data_dir, "all.fasta")
records = []
for _, row in Seqs.iterrows():
    record = SeqRecord(Seq(row["seq"]), id=str(row["id"]), description="", name="")
    records.append(record)
SeqIO.write(records, all_fasta, "fasta")

In [ ]:
cluster40 = os.path.join(cluster_data_dir, "all_clstr40")

In [ ]:
os.system(
    f"cd-hit -i {all_fasta} -o {cluster40} -c 0.4 -n 2 -d 0 -aS 0.9 -aL 0.9 -g 1 -T 16"
)

In [ ]:
def read_clstr(file):
    with open(file, "r") as f:
        lines = f.readlines()
    clusters = {}
    cluster_cnt = 0
    protein_cnt = 0
    for line in lines:
        if line.startswith(">"):
            cluster_id = line[1:].strip()
            clusters[cluster_id] = {}
            cluster_cnt += 1
            cluster_prot_cnt = 0
        else:
            m = re.match(r"\d+\s+(\d+)aa, >(\d+)\.\.\.", line)
            protein_len = int(m.group(1))
            protein_id = int(m.group(2))
            clusters[cluster_id][f"Protein {cluster_prot_cnt}"] = {
                "id": protein_id,
                "len": protein_len,
            }
            protein_cnt += 1
            cluster_prot_cnt += 1
    return clusters, cluster_cnt, protein_cnt

In [ ]:
cluster_dict_all, cluster_count, protein_count = read_clstr(cluster40 + ".clstr")
print(f"Cluster count: {cluster_count}")
print(f"Protein count: {protein_count}")

In [ ]:
np.random.seed(42)
cluster_ids = np.arange(0, cluster_count)
np.random.shuffle(cluster_ids)
cluster_ids

In [ ]:
fold_count = 10
cluster_ids_10fold = np.array_split(cluster_ids, fold_count)
seq_fasta = SeqIO.index(all_fasta, "fasta")
for i, fold in enumerate(cluster_ids_10fold):
    prot_id_list = []
    for cluster_id in fold:
        for _, protein in cluster_dict_all["Cluster " + str(cluster_id)].items():
            prot_id_list.append(protein["id"])
    prot_id_list = list(set(prot_id_list))
    prot_id_list.sort()
    fold_seq_fasta = [seq_fasta[str(prot_id)] for prot_id in prot_id_list]
    fold_fasta = os.path.join(cluster_data_dir, f"fold{i}.fasta")
    SeqIO.write(fold_seq_fasta, fold_fasta, "fasta")
    print(f"Fold {i}: {fold.shape} clusters, {len(prot_id_list)} proteins")

In [ ]:
for i in range(fold_count):
    pre = os.path.join(cluster_data_dir, f"fold{i}")
    os.system(
        f"cd-hit -i {pre + '.fasta'} -o {pre + '_clstr90'} -c 0.9 -n 5 -d 0  -aS 0.9 -aL 0.9 -T 16"
    )

In [ ]:
fold_dict = {}
fold_list = []
for i in range(fold_count):
    clstr = os.path.join(cluster_data_dir, f"fold{i}_clstr90.clstr")
    cluster_dict, cluster_count, protein_count = read_clstr(clstr)
    fold_dict[f"Fold {i}"] = cluster_dict
    cluster_list = []
    for cluster_id, protein_id_s in cluster_dict.items():
        protein_list = []
        for protein_id_, protein in protein_id_s.items():
            row = Seqs[Seqs["id"] == protein["id"]]
            NCBI_ids = row["NCBI_protein_ids"].values[0]
            AFDB_ids = row["AFDB_protein_ids"].values[0]
            protein_list.append((protein["id"], protein["len"], NCBI_ids, AFDB_ids))
        cluster_list.append(protein_list)
    fold_list.append(cluster_list)
    print(f"Fold {i}: {cluster_count} clusters, {protein_count} proteins")

In [ ]:
with open(os.path.join(data_dir, "fold_dict.json"), "w") as f:
    json.dump(fold_dict, f, indent=4)
with open(os.path.join(data_dir, "fold_list.pkl"), "wb") as f:
    pickle.dump(fold_list, f)
with open(os.path.join(data_dir, "fold_list.json"), "w") as f:
    json.dump(fold_list, f, indent=4)
print("Done")

In [ ]:
# fold_list: List[List[List[Tuple[int, int, List[int], List[int]]]]]], all 10 folds
# len(fold_list) == 10, the same as fold_count
#
# fold_list[i]: List[List[Tuple[int, int, List[int], List[int]]]], the i-th fold
# len(fold_list[i]) == cluster_count, the same as the number of clusters in fold i
#
# fold_list[i][j]: List[Tuple[int, int, List[int], List[int]]], the j-th cluster in fold i
# len(fold_list[i][j]) == number of proteins in cluster j in fold i
#
# fold_list[i][j][k]: Tuple[int, int, List[int], List[int]], the k-th protein in cluster j in fold i
# fold_list[i][j][k][0]: int, the id of protein k in cluster j in fold i
# fold_list[i][j][k][1]: int, the length of protein k in cluster j in fold i
# fold_list[i][j][k][2]: List[int], NCBI ids with the same seq as protein k in cluster j in fold i
# fold_list[i][j][k][3]: List[int], AFDB ids with the same seq as protein k in cluster j in fold i
with open(os.path.join(data_dir, "fold_list.pkl"), "rb") as f:
    x = pickle.load(f)
x[0][0][0]